# Supervisor Workflow: AI Tutor Routing Student Questions

This notebook demonstrates a supervisor-style workflow.

Scenario:

A student asks a question. A supervisor step classifies the question and routes it to the most appropriate specialist.

Specialists:

- Concept Explanation Agent
- Coding Help Agent
- Math Derivation Agent
- Study Strategy Agent

## Setup

In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)



from picoagents.workflow import Workflow, WorkflowRunner, FunctionStep
from picoagents.workflow.core import WorkflowMetadata, StepMetadata, Context

runner = WorkflowRunner()

API key loaded successfully.


## Define models and routing functions

In [15]:
class StudentQuestionInput(BaseModel):
    """Input schema for an incoming student question."""
    question: str


class RoutingOutput(BaseModel):
    """Supervisor routing decision and rationale."""
    question: str
    route: Literal["concept", "coding", "math", "study"]
    reason: str


class TutorAnswerOutput(BaseModel):
    """Final specialist response payload."""
    result: str


async def route_student_question(input_data: StudentQuestionInput, context: Context) -> RoutingOutput:
    """Classify the question and select the specialist route."""
    q = input_data.question.lower()

    if any(word in q for word in ["python", "code", "sklearn", "error", "notebook"]):
        return RoutingOutput(question=input_data.question, route="coding", reason="The question asks for coding help.")

    if any(word in q for word in ["derive", "equation", "gradient", "loss", "probability"]):
        return RoutingOutput(question=input_data.question, route="math", reason="The question asks for mathematical derivation.")

    if any(word in q for word in ["study", "prepare", "exam", "learn", "practice"]):
        return RoutingOutput(question=input_data.question, route="study", reason="The question asks for study strategy.")

    return RoutingOutput(question=input_data.question, route="concept", reason="The question asks for conceptual explanation.")


async def concept_explanation(input_data: RoutingOutput, context: Context) -> TutorAnswerOutput:
    """Return a concept-focused tutoring response."""
    return TutorAnswerOutput(result=f"Concept Explanation Agent: I will explain the core concept behind: {input_data.question}")


async def coding_help(input_data: RoutingOutput, context: Context) -> TutorAnswerOutput:
    """Return a coding-oriented tutoring response."""
    return TutorAnswerOutput(result=f"Coding Help Agent: I will provide a scikit-learn example for: {input_data.question}")


async def math_derivation(input_data: RoutingOutput, context: Context) -> TutorAnswerOutput:
    """Return a math-derivation tutoring response."""
    return TutorAnswerOutput(result=f"Math Derivation Agent: I will derive the mathematical steps for: {input_data.question}")


async def study_strategy(input_data: RoutingOutput, context: Context) -> TutorAnswerOutput:
    """Return a study-planning tutoring response."""
    return TutorAnswerOutput(result=f"Study Strategy Agent: I will suggest a study plan for: {input_data.question}")

## Build supervisor workflow

### Workflow Diagram: add_step vs add_edge

`add_step(...)` creates nodes in the workflow graph.
`add_edge(from, to, condition=...)` creates conditional routing arrows between nodes.

```mermaid
flowchart
    A[tutor_supervisor] -->|route = concept| B[concept_agent]
    A -->|route = coding| C[coding_agent]
    A -->|route = math| D[math_agent]
    A -->|route = study| E[study_agent]
```

Quick code mapping:
- `add_step(...)`: register `router_step` and each specialist step as nodes.
- `add_edge(...)`: define routing logic from supervisor to each specialist.
- `set_start_step("tutor_supervisor")`: marks the entry node.

In [16]:
router_step = FunctionStep(
    step_id="tutor_supervisor",
    metadata=StepMetadata(name="Tutor Supervisor"),
    input_type=StudentQuestionInput,
    output_type=RoutingOutput,
    func=route_student_question
)

concept_step = FunctionStep(
    step_id="concept_agent",
    metadata=StepMetadata(name="Concept Explanation Agent"),
    input_type=RoutingOutput,
    output_type=TutorAnswerOutput,
    func=concept_explanation
)

coding_step = FunctionStep(
    step_id="coding_agent",
    metadata=StepMetadata(name="Coding Help Agent"),
    input_type=RoutingOutput,
    output_type=TutorAnswerOutput,
    func=coding_help
)

math_step = FunctionStep(
    step_id="math_agent",
    metadata=StepMetadata(name="Math Derivation Agent"),
    input_type=RoutingOutput,
    output_type=TutorAnswerOutput,
    func=math_derivation
)

study_step = FunctionStep(
    step_id="study_agent",
    metadata=StepMetadata(name="Study Strategy Agent"),
    input_type=RoutingOutput,
    output_type=TutorAnswerOutput,
    func=study_strategy
)

supervisor_workflow = (
    Workflow(metadata=WorkflowMetadata(name="Supervisor AI Tutor Routing"))
    # add_step: register nodes in the workflow graph
    .add_step(router_step)
    .add_step(concept_step)
    .add_step(coding_step)
    .add_step(math_step)
    .add_step(study_step)
    # add_edge: route from supervisor to specialists based on route output
    .add_edge("tutor_supervisor", "concept_agent", condition={
        "type": "output_based", "field": "route", "operator": "==", "value": "concept"
    })
    .add_edge("tutor_supervisor", "coding_agent", condition={
        "type": "output_based", "field": "route", "operator": "==", "value": "coding"
    })
    .add_edge("tutor_supervisor", "math_agent", condition={
        "type": "output_based", "field": "route", "operator": "==", "value": "math"
    })
    .add_edge("tutor_supervisor", "study_agent", condition={
        "type": "output_based", "field": "route", "operator": "==", "value": "study"
    })
    .set_start_step("tutor_supervisor")
    .add_end_step("concept_agent")
    .add_end_step("coding_agent")
    .add_end_step("math_agent")
    .add_end_step("study_agent")
)

## Run examples

In [17]:
questions = [
    {"question": "Can you explain overfitting in simple terms?"},
    {"question": "How do I use GridSearchCV in scikit-learn?"},
    {"question": "Can you derive the logistic regression loss?"},
    {"question": "How should I prepare for the machine learning exam?"}
]

for q in questions:
    print("\nQUESTION:", q["question"])
    async for event in runner.run_stream(supervisor_workflow, q):
        print(event)


QUESTION: Can you explain overfitting in simple terms?
[21:04:23] 🚀 Workflow started with input: {'question': 'Can you explain overfitting in simple terms?'}
[21:04:23] ▶️  Step 'tutor_supervisor' started
[21:04:23] ✅ Step 'tutor_supervisor' completed → {'question': 'Can you explain overfitting in simple terms?', 'route': 'concept', 'reason': 'The question asks for conceptual explanation.'}
[21:04:23] 🔗 tutor_supervisor → concept_agent
[21:04:23] 🔗 tutor_supervisor → coding_agent
[21:04:23] 🔗 tutor_supervisor → math_agent
[21:04:23] 🔗 tutor_supervisor → study_agent
[21:04:23] ▶️  Step 'concept_agent' started
[21:04:23] ✅ Step 'concept_agent' completed → {'result': 'Concept Explanation Agent: I will explain the core concept behind: Can you explain overfitting in simple terms?'}
[21:04:23] ✅ Workflow completed in 0.00s (2 steps)

QUESTION: How do I use GridSearchCV in scikit-learn?
[21:04:23] 🚀 Workflow started with input: {'question': 'How do I use GridSearchCV in scikit-learn?'}
[21:0

## Reflection questions

1. Why is this called a supervisor workflow?
2. How could the rule-based router be replaced by an LLM-based classifier?
3. What is the advantage of keeping the router deterministic?